# Preprocessing — Transformaciones de datos

**Objetivo:** Transformar el dataset enriquecido en datos numéricos, sin nulos, listos para análisis.

En este notebook:
1. Cargamos el dataset de feature engineering
2. Limpiamos basura estadística (XNA en CODE_GENDER)
3. Imputamos nulos con estrategia definitiva
4. Encoding de categóricas (estrategia definida)
5. Guardamos datos transformados

**Estrategia de encoding:**
- Binarias → Label Encoding
- Pocas categorías (<10) → One-Hot Encoding
- EDUCATION_TYPE → Ordinal Encoding
- Alta cardinalidad (OCCUPATION, ORGANIZATION, WEEKDAY) → Frequency Encoding

**NOTA:** El train/test split y el scaling van en `08_modeling.ipynb`.

**Regla:** Este notebook transforma TODO el dataset. Los test estadísticos necesitan todos los datos.

---
## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print('Setup listo.')

Setup listo.


---
## 2. Carga de datos

In [2]:
df = pd.read_csv('../data/processed/application_train_features.csv')

print(f'Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Target:')
print(df['TARGET'].value_counts())

Dataset: 307,511 filas x 185 columnas
Target:
TARGET
0    282686
1     24825
Name: count, dtype: int64


---
## 3. Limpiar basura estadística

In [3]:
# CODE_GENDER: eliminar XNA (4 instancias de 307K)
print(f'Antes: {df["CODE_GENDER"].value_counts().to_dict()}')
df['CODE_GENDER'] = df['CODE_GENDER'].replace('XNA', np.nan)
print(f'Después: {df["CODE_GENDER"].value_counts().to_dict()}')

Antes: {'F': 202448, 'M': 105059, 'XNA': 4}
Después: {'F': 202448, 'M': 105059}


---
## 4. Identificar tipos de columnas

In [4]:
TARGET_COL = 'TARGET'
ID_COL = 'SK_ID_CURR'

num_cols = df.drop(columns=[ID_COL, TARGET_COL]).select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.drop(columns=[ID_COL, TARGET_COL]).select_dtypes(exclude=[np.number]).columns.tolist()

print(f'Numéricas: {len(num_cols)}')
print(f'Categóricas: {len(cat_cols)}')
print()
for c in cat_cols:
    print(f'  {c:40s} → {df[c].nunique()} categorías')

Numéricas: 167
Categóricas: 16

  NAME_CONTRACT_TYPE                       → 2 categorías
  CODE_GENDER                              → 2 categorías
  FLAG_OWN_CAR                             → 2 categorías
  FLAG_OWN_REALTY                          → 2 categorías
  NAME_TYPE_SUITE                          → 7 categorías
  NAME_INCOME_TYPE                         → 8 categorías
  NAME_EDUCATION_TYPE                      → 5 categorías
  NAME_FAMILY_STATUS                       → 6 categorías
  NAME_HOUSING_TYPE                        → 6 categorías
  OCCUPATION_TYPE                          → 18 categorías
  WEEKDAY_APPR_PROCESS_START               → 7 categorías
  ORGANIZATION_TYPE                        → 58 categorías
  FONDKAPREMONT_MODE                       → 4 categorías
  HOUSETYPE_MODE                           → 3 categorías
  WALLSMATERIAL_MODE                       → 7 categorías
  EMERGENCYSTATE_MODE                      → 2 categorías


---
## 5. Imputación de nulos

Estrategia:
- **Numéricas:** mediana (robusta a outliers)
- **Categóricas:** moda (valor más frecuente)

In [5]:
# Nulos antes de imputar
nulls_before = df.isnull().sum()
nulls_before = nulls_before[nulls_before > 0].sort_values(ascending=False)

print(f'Columnas con nulos: {len(nulls_before)} de {df.shape[1]}')
print(f'Total nulos: {nulls_before.sum():,}')
print()
print('Top 10:')
for col, n in nulls_before.head(10).items():
    print(f'  {col:40s} → {n:>8,} ({n/len(df)*100:.1f}%)')

Columnas con nulos: 124 de 185
Total nulos: 11,179,538

Top 10:
  AMT_ANNUITY_mean_x                       →  227,502 (74.0%)
  AMT_ANNUITY_max_x                        →  227,502 (74.0%)
  COMMONAREA_AVG                           →  214,865 (69.9%)
  COMMONAREA_MODE                          →  214,865 (69.9%)
  COMMONAREA_MEDI                          →  214,865 (69.9%)
  NONLIVINGAPARTMENTS_MODE                 →  213,514 (69.4%)
  NONLIVINGAPARTMENTS_MEDI                 →  213,514 (69.4%)
  NONLIVINGAPARTMENTS_AVG                  →  213,514 (69.4%)
  FONDKAPREMONT_MODE                       →  210,295 (68.4%)
  LIVINGAPARTMENTS_MODE                    →  210,199 (68.4%)


In [6]:
# Imputar numéricas con mediana
imputation_medians = {}
for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        imputation_medians[col] = median_val
        df[col] = df[col].fillna(median_val)

# Imputar categóricas con moda
imputation_modes = {}
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        imputation_modes[col] = mode_val
        df[col] = df[col].fillna(mode_val)

# Verificar
nulls_after = df.isnull().sum().sum()
print(f'Nulos después de imputar: {nulls_after}')
print(f'Medianas guardadas: {len(imputation_medians)}')
print(f'Modas guardadas: {len(imputation_modes)}')

Nulos después de imputar: 0
Medianas guardadas: 117
Modas guardadas: 7


---
## 6. Definir estrategia de encoding

| Variable | Categorías | Estrategia |
|---|---|---|
| NAME_CONTRACT_TYPE | 2 | One-Hot drop first |
| CODE_GENDER | 2 (M/F) | One-Hot drop first |
| FLAG_OWN_CAR | 2 | One-Hot drop first |
| FLAG_OWN_REALTY | 2 | One-Hot drop first |
| NAME_TYPE_SUITE | 7 | One-Hot |
| NAME_INCOME_TYPE | 8 | One-Hot |
| NAME_EDUCATION_TYPE | 5 | Ordinal |
| NAME_FAMILY_STATUS | 6 | One-Hot |
| NAME_HOUSING_TYPE | 6 | One-Hot |
| OCCUPATION_TYPE | 18 | Frequency Encoding |
| WEEKDAY_APPR_PROCESS_START | 7 | Frequency Encoding |
| ORGANIZATION_TYPE | 58 | Frequency Encoding |
| FONDKAPREMONT_MODE | 4 | One-Hot |
| HOUSETYPE_MODE | 3 | One-Hot |
| WALLSMATERIAL_MODE | 7 | One-Hot |
| EMERGENCYSTATE_MODE | 2 | One-Hot drop first |

In [7]:
# Definir grupos de encoding

# One-Hot drop first (binarias)
oh_drop_first = [
    'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 
    'FLAG_OWN_REALTY', 'EMERGENCYSTATE_MODE'
]

# One-Hot normal (multi-categoría)
oh_normal = [
    'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE',
    'WALLSMATERIAL_MODE'
]

# Ordinal (tiene orden natural)
ordinal_cols = ['NAME_EDUCATION_TYPE']

# Frequency Encoding (alta cardinalidad)
freq_cols = ['OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE']

print('Estrategia definida.')
print(f'  One-Hot drop first: {len(oh_drop_first)} columnas')
print(f'  One-Hot normal:     {len(oh_normal)} columnas')
print(f'  Ordinal:            {len(ordinal_cols)} columnas')
print(f'  Frequency Encoding: {len(freq_cols)} columnas')

Estrategia definida.
  One-Hot drop first: 5 columnas
  One-Hot normal:     7 columnas
  Ordinal:            1 columnas
  Frequency Encoding: 3 columnas


---
## 7. Frequency Encoding (alta cardinalidad)

Reemplaza cada categoría con su frecuencia relativa. Sin data leakage.

In [8]:
for col in freq_cols:
    freq = df[col].value_counts(normalize=True)
    df[col] = df[col].map(freq)
    print(f'{col:40s} → {freq.shape[0]} categorías → 1 columna numérica')

print('\nFrequency Encoding completado.')

OCCUPATION_TYPE                          → 18 categorías → 1 columna numérica
WEEKDAY_APPR_PROCESS_START               → 7 categorías → 1 columna numérica
ORGANIZATION_TYPE                        → 58 categorías → 1 columna numérica

Frequency Encoding completado.


---
## 8. Ordinal Encoding (NAME_EDUCATION_TYPE)

Orden definido por significado semántico.

In [9]:
# Definir orden explícito
education_order = {
    'Lower secondary': 0,
    'Secondary / secondary special': 1,
    'Incomplete higher': 2,
    'Higher education': 3,
    'Academic degree': 4
}

for col in ordinal_cols:
    df[col] = df[col].map(education_order)
    print(f'{col}:')
    for cat, val in education_order.items():
        count = (df[col] == val).sum()
        print(f'  {val} → {cat:40s} ({count:>8,} muestras)')

print('\nOrdinal Encoding completado.')

NAME_EDUCATION_TYPE:
  0 → Lower secondary                          (   3,816 muestras)
  1 → Secondary / secondary special            ( 218,391 muestras)
  2 → Incomplete higher                        (  10,277 muestras)
  3 → Higher education                         (  74,863 muestras)
  4 → Academic degree                          (     164 muestras)

Ordinal Encoding completado.


---
## 9. One-Hot Encoding

In [10]:
# One-Hot drop first
df = pd.get_dummies(df, columns=oh_drop_first, drop_first=True)
print(f'One-Hot drop first: {len(oh_drop_first)} columnas → {df.shape[1]} totales')

# One-Hot normal
df = pd.get_dummies(df, columns=oh_normal, drop_first=False)
print(f'One-Hot normal:     {len(oh_normal)} columnas → {df.shape[1]} totales')

# Castear columnas one-hot de bool a int64
oh_cols = [c for c in df.columns if c not in [ID_COL, TARGET_COL] and df[c].dtype == bool]
df[oh_cols] = df[oh_cols].astype(int)
print(f'\nColumnas one-hot casteadas a int64: {len(oh_cols)}')

One-Hot drop first: 5 columnas → 185 totales
One-Hot normal:     7 columnas → 219 totales

Columnas one-hot casteadas a int64: 46


In [11]:
# Verificar que no quedan categóricas
remaining_cat = df.drop(columns=[ID_COL, TARGET_COL]).select_dtypes(exclude=[np.number]).columns.tolist()
print(f'Categóricas restantes: {len(remaining_cat)}')
if remaining_cat:
    print(remaining_cat)

Categóricas restantes: 0


---
## 10. Resumen final

In [12]:
print('RESUMEN DEL PREPROCESSING')
print('=' * 50)
print(f'Features finales:  {df.shape[1] - 2}')
print(f'Muestras:          {df.shape[0]:,}')
print(f'Nulos restantes:   {df.isnull().sum().sum()}')
print()
print('Transformaciones aplicadas:')
print(f'  - Limpieza: XNA en CODE_GENDER imputado con moda')
print(f'  - Imputación: mediana ({len(imputation_medians)} cols) / moda ({len(imputation_modes)} cols)')
print(f'  - Frequency Encoding: {len(freq_cols)} columnas (sin leakage)')
print(f'  - Ordinal Encoding: {len(ordinal_cols)} columna(s)')
print(f'  - One-Hot drop first: {len(oh_drop_first)} columnas')
print(f'  - One-Hot normal: {len(oh_normal)} columnas')

RESUMEN DEL PREPROCESSING
Features finales:  217
Muestras:          307,511
Nulos restantes:   0

Transformaciones aplicadas:
  - Limpieza: XNA en CODE_GENDER imputado con moda
  - Imputación: mediana (117 cols) / moda (7 cols)
  - Frequency Encoding: 3 columnas (sin leakage)
  - Ordinal Encoding: 1 columna(s)
  - One-Hot drop first: 5 columnas
  - One-Hot normal: 7 columnas


---
## 11. Guardar datos transformados

In [13]:
import os
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/application_train_preprocessed.csv', index=False)

print(f'Guardado: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Ruta: data/processed/application_train_preprocessed.csv')

Guardado: 307,511 filas x 219 columnas
Ruta: data/processed/application_train_preprocessed.csv
